# Seconda lezione: From samples to a musical language


In [1]:
from IPython.display import Audio

In [3]:
# With this library, writing "sd.play()" you'll be able to play sounds
import sounddevice as sd
# But it's not our objective right now

We can create list that hold values this way.

In [4]:
wave = ([0] * 400 + [1] * 400) * 100

"normalize" in the Audio function allows our wave to be between -1 and 1; if it is not, then normalize=True is necessary

In [5]:
Audio(wave, rate=48000, normalize=False)

If 48000 is the sample rate (SR), then $T = \frac{NumOfSamples}{SR}$

In [7]:
n_samples = 800
duration_of_a_period = n_samples/48000 

Therefore, the frequency becomes $f = \frac{1}{T}$.

We want to create an implementation that with general variables allows us to create a wave with specific properties. For instance, with:
- frequency,
- sample rate,
- amplitude and
- duration

we can calculate different quantities that allows us to create a general way of representing a wave:

In [16]:
f = 440 # in Hz
sample_rate = 48000
amplitude = .15
duration = 1

period_duration = 1/f #in seconds
period_samples = period_duration * sample_rate
half_period_samples = period_samples/2
number_of_periods = duration/period_duration

wave1 = ([-amplitude] * round(half_period_samples) + [amplitude] * round(half_period_samples)) * round(number_of_periods)
Audio(wave, rate=sample_rate, normalize=False)

Since notes go up an octave by doubling, the way we get a note a number $n$ of semitones above the fundamental frequency we must use this equation:
$$f_2 = f_1 \times 2^(\frac{n}{12})$$
This way, we can create a multiple notes after the first one.

In [17]:
f = 440 * 2 ** (2 / 12) # in Hz
sample_rate = 48000
amplitude = .15
duration = 1

period_duration = 1/f #in seconds
period_samples = period_duration * sample_rate
half_period_samples = period_samples/2
number_of_periods = duration/period_duration

wave2 = ([-amplitude] * round(half_period_samples) + [amplitude] * round(half_period_samples)) * round(number_of_periods)
Audio(wave1+wave2, rate=sample_rate, normalize=False)

We could also create a third note

In [18]:
f = 440 * 2 ** (4 / 12) # in Hz
sample_rate = 48000
amplitude = .15
duration = 1

period_duration = 1/f #in seconds
period_samples = period_duration * sample_rate
half_period_samples = period_samples/2
number_of_periods = duration/period_duration

wave3 = ([-amplitude] * round(half_period_samples) + [amplitude] * round(half_period_samples)) * round(number_of_periods)
Audio(wave1+wave2+wave3, rate=sample_rate, normalize=False)

But there is a much better way to do it: **FUNCTIONS**

Functions require the definition of the input arguments, to which you can give default parameters if needed.  
Function definition are given by
```
def name_of_function(variable = default_value):
```
And if required there must be a return at the end.

The variables inside the parenthesis after the name of the function are called _parameters_, the actual value they're given is called _argument_.

In [20]:
def square_wave(f = 440,
                sample_rate = 48000,
                amplitude = .15,
                duration = .5):

    period_duration = 1/f #in seconds
    period_samples = period_duration * sample_rate
    half_period_samples = period_samples/2
    number_of_periods = duration/period_duration
    
    return ([-amplitude] * round(half_period_samples) + 
            [amplitude] * round(half_period_samples)) * round(number_of_periods)

In this way we can just call it and

In [21]:
Audio(square_wave(f=440*2**(2/12)),rate=48000,normalize=False)

We could now create a melody much, much faster and less error-prone just summing subsequent waves created through the function.  
If we create something like this
```
opening = [square_wave(440*2**(n/12) for n in [0, 2, 4, 0] * 2]
```
the object we're passing as argument is a list of lists, not just a list! Therefore we should just sum all the elements, but it's not efficient.  
What we should do is, since we're using lists and not numbers (if you do it, you get an error because the sum function initializes a temporary variable to 0 and you'd be trying to sum lists to variables), using the sum function but specifing the type of initial value:

In [24]:
opening = [square_wave(440*2**(n/12)) for n in [0, 2, 4, 0] * 2]
opening_wave = sum(opening, [])
Audio(opening_wave, rate=48000, normalize=False)

This way, we can create melodies kind of easily!

In [32]:
beat_duration = 0.4
[0,0,2,2,4,4,0,0]*2+[4,4,5,5,7,7,7,7]*2+[7,9,7,5,4,4,0,0]*2+[0,0,-5,-5,0,0,0,0]*2
fra_martins1 = [square_wave(f=440*2**(n/12),duration=beat_duration) for n in [0,2,4,0]*2+[4,5,7,7]*2]
fra_martins2 = [square_wave(f=440*2**(n/12),duration=beat_duration/2) for n in [7,9,7,5,4,4,0,0]*2]
fra_martins3 = [square_wave(f=440*2**(n/12),duration=beat_duration) for n in [0,-5,0,0]*2]
fra_martins = fra_martins1 + fra_martins2 + fra_martins3
fra_martins_wave = sum(fra_martins, [])
Audio(fra_martins_wave, rate=48000, normalize=False)

Or we can play Fra Martino establishing the main beat as the eigth note:

In [37]:
beat_duration_2 = 0.2
fra_martins_2 = [square_wave(440 * 2 ** (n/12), duration=beat_duration_2) for n in 
                 [0,0,2,2,4,4,0,0]*2+[4,4,5,5,7,7,7,7]*2+[7,9,7,5,4,4,0,0]*2+[0,0,-5,-5,0,0,0,0]*2]
fra_martins_2_wave = sum(fra_martins_2, [])
Audio(fra_martins_2_wave, rate=48000, normalize=False)

Or you could, as the professor did, create a list of lists where each internal list is
```
[note, duration]
```

In [38]:
# onda = [[nota, durata], [nota, durata]...   ]
onda = [[0,.5],[5,.6],[10,.7],[15,.8]]
onda_lista = [square_wave(f = 440 * 2 ** (n[0]/12),duration = n[1]) for n in onda]
onda_wave = sum(onda_lista, [])
Audio(onda_wave, rate=48000, normalize=False)

If we put $note=-100$ we expect a note so low we can't hear it, creating a sort of a pause. When one tries to do it, it doesn't work.  
How can we do it? Audio has a "gate" parameter that tells the computer how much a note should be played. For instance, gate=0.9 means that for 90% of the duration the note will play, and for 10% it doesn't.  
In this way we could create a "silence wave function", which will have a specific gate!

In [41]:
def silence_wave(f = 440,
                sample_rate = 48000,
                amplitude = .15,
                duration = .5):
    return ([0] * round(duration * sample_rate))

In [51]:
Audio(silence_wave(duration = 2),rate=48000,normalize=False)

In [57]:
melody = [[0,1,square_wave],[0,1,silence_wave],[2,1,square_wave]]
waves = [n[2](f=440*2**(n[0]/12),duration=n[1]) for n in melody]
Audio(sum(waves, []), rate=48000, normalize=False)

### Percussive sounds
How do we create a sound which is much less regular? We get noise! How do we make noise in python? With pseudo number generators or with the random library.

In [52]:
from random import random

With this library, for instance, we can create **WHITE NOISE**

In [54]:
signal = [random() for n in range(48000)]
Audio(signal, rate=48000,normalize=False)

We could create a high-hat! Alternating silence with noise we get something similar but doesn't sound like a good hi-hat.

In [55]:
# bad high-hat
signal = [random() for n in range(4800)] + silence_wave(duration=0.1)
Audio(signal*10, normalize=False, rate=48000)

In [56]:
# better high-hat
signal_hh = [random() * 1 - (n / 4800) for n in range(4800)] + silence_wave(duration=0.1)
Audio(signal_hh*10, rate=48000)

The factor $1-\frac{n}{4800}$ supposedly can make the noise go to 0 and make it a little less harsh.  
Now work in 2 and put the high-hat in one computer and the other with Fra Martino.

In [111]:
suono1 = square_wave(f=440)
suono2 = square_wave(f=440*2**(2/12))
mega_suono = [x+y for x,y in zip(suono1,suono2)]
Audio(mega_suono*10, normalize=False, rate=48000)


In [114]:
eight_duration = 0.2

frere_jaques = [square_wave(440 * 2 ** (n/12), duration=eight_duration) for n in 
                 [0,0,2,2,4,4,0,0]*2+[4,4,5,5,7,7,7,7]*2+[7,9,7,5,4,4,0,0]*2+[0,0,-5,-5,0,0,0,0]*2]
frere_jaques_wave = sum(frere_jaques, [])

hh = [random()*1-n/len(frere_jaques[0]) for n in range(len(frere_jaques[0]))]

big_wave = [x+y for x,y in zip(frere_jaques_wave,hh)]
Audio(big_wave, rate=48000, normalize=True)